# CareerPilot AI — 3. Job Matching (RAG retrieval)

Loads the most recently saved profile from `careerpilot.db` and the postings from `job_dataset.json`, embeds both with a pre-trained model, and ranks jobs by cosine similarity. This is retrieval, not training — nothing here learns anything, we're just comparing pre-trained embeddings.

**Requires:** `1_resume_parsing.ipynb` has been run at least once (so `careerpilot.db` has a profile), and `2_job_fetching.ipynb` has been run at least once (so `job_dataset.json` exists).

**Fixed in this version:** `job_dataset.json` is now fetched per-profile (see `2_job_fetching.ipynb`), so this notebook checks whether the job pool it's about to rank was actually fetched for the *current* candidate, and warns loudly if not — the same pattern `5_skill_gap_analysis.ipynb` already uses for `top_matches.json`.

**Output:** prints the top 5 matches and saves them to `top_matches.json`, which `4_generate_explanations.ipynb` reads next.


## Step 1 — Setup


In [7]:
!pip install sentence-transformers --quiet


In [8]:
import json
import sqlite3
import numpy as np
from numpy.linalg import norm
from sentence_transformers import SentenceTransformer

DB_PATH = "careerpilot.db"
JOB_DATASET_PATH = "job_dataset.json"
TOP_MATCHES_PATH = "top_matches.json"


## Step 2 — Load the latest saved profile from SQLite


In [9]:
def load_latest_profile() -> dict:
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM career_profiles ORDER BY id DESC LIMIT 1")
        row = cur.fetchone()
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()

    if row is None:
        raise ValueError("No profiles found in careerpilot.db — run 1_resume_parsing.ipynb first.")

    profile = dict(zip(cols, row))
    for f in ["skills", "education", "experience", "organizations", "certifications"]:
        profile[f] = json.loads(profile[f]) if profile[f] else []
    return profile

profile = load_latest_profile()
profile_id = profile["id"]
print(f"Loaded profile #{profile_id}: {profile.get('name') or profile.get('filename')}")
print(f"Skills: {profile['skills']}")


Loaded profile #5: linkedin.com/in/sivateja-somisetty •
Skills: ['CSS', 'FastAPI', 'Git', 'Java', 'JavaScript', 'Machine Learning', 'Python']


## Step 3 — Load job postings


In [10]:
with open(JOB_DATASET_PATH, "r") as f:
    job_data = json.load(f)

# 2_job_fetching.ipynb now saves {"profile_id", "generated_at", "jobs": [...]} so the
# job pool can be checked for freshness against the current profile. Still accept the
# old bare-list format for backward compatibility.
if isinstance(job_data, list):
    jobs = job_data
    job_dataset_profile_id = None
else:
    jobs = job_data.get("jobs", [])
    job_dataset_profile_id = job_data.get("profile_id")

print(f"Loaded {len(jobs)} job postings.")

if job_dataset_profile_id is not None and job_dataset_profile_id != profile_id:
    print(
        f"\u26a0\ufe0f  WARNING: job_dataset.json was fetched for profile #{job_dataset_profile_id}, "
        f"but the latest profile in the database is now #{profile_id} "
        f"({profile.get('name') or profile.get('filename')}).\n"
        f"    A different resume was likely uploaded/re-parsed since 2_job_fetching.ipynb last ran, "
        f"so this job pool (and its search terms) reflect the wrong candidate.\n"
        f"    Re-run 2_job_fetching.ipynb to fetch postings tailored to the current resume.\n"
    )


Loaded 44 job postings.


## Step 4 — Embed and rank by cosine similarity

`all-MiniLM-L6-v2` downloads once (a couple hundred MB, may take a minute or two the first time) — don't run other cells while it's working.


In [11]:
def profile_to_text(p: dict) -> str:
    skills_text = ", ".join(p.get("skills", []))
    return f"Skills: {skills_text}. Background: {p.get('raw_text', '')[:1000]}"

def job_to_text(job: dict) -> str:
    skills_text = ", ".join(job.get("skills", []))
    return f"{job['title']} at {job['company']}. Required skills: {skills_text}. {job['description']}"

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (norm(a) * norm(b)))

model = SentenceTransformer("all-MiniLM-L6-v2")

profile_vector = model.encode(profile_to_text(profile))
job_texts = [job_to_text(job) for job in jobs]
job_vectors = model.encode(job_texts)

scored = [(cosine_similarity(profile_vector, vec), job) for job, vec in zip(jobs, job_vectors)]
scored.sort(key=lambda x: x[0], reverse=True)
top_matches = scored[:5]

print("Top matches:\n")
for rank, (score, job) in enumerate(top_matches, start=1):
    print(f"{rank}. {job['title']} — {job['company']}  (score: {score:.3f})")
    print(f"   Required skills: {', '.join(job['skills'])}")
    print(f"   Apply: {job.get('apply_link', 'link not available')}")
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top matches:

1. Python Backend Developer Intern (FastAPI) — appscrip  (score: 0.680)
   Required skills: FastAPI, Python, R
   Apply: https://www.adzuna.in/details/5812187923?utm_medium=api&utm_source=6abd39fa

2. Senior Backend Engineer - FastAPI — LearnTube.ai  (score: 0.652)
   Required skills: AWS, CI/CD, Docker, FastAPI, GCP, Git, MongoDB, PostgreSQL, Python, R, REST API, SQL
   Apply: https://www.adzuna.in/details/5770458005?utm_medium=api&utm_source=6abd39fa

3. Full Stack Python Developer (FastAPI) — Topcoder  (score: 0.633)
   Required skills: FastAPI, Python, R, REST API
   Apply: https://www.adzuna.in/details/5671080668?utm_medium=api&utm_source=6abd39fa

4. Javascript Developer — Extern Labs  (score: 0.623)
   Required skills: Angular, CSS, Excel, Git, HTML, Java, JavaScript, R, React
   Apply: https://www.adzuna.in/details/1914318449?utm_medium=api&utm_source=6abd39fa

5. Python Developer (FastAPI) — Hunarstreet Technologies Pvt Ltd  (score: 0.617)
   Required skills: Fas

## Step 5 — Save results for the explanation step


In [12]:
output = {
    "profile_id": profile_id,
    "matches": [
        {"score": score, **job} for score, job in top_matches
    ],
}

with open(TOP_MATCHES_PATH, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {TOP_MATCHES_PATH} — ready for 4_generate_explanations.ipynb")


Saved top_matches.json — ready for 4_generate_explanations.ipynb
